In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")

Koneksi ke MongoDB berhasil!


In [2]:
menu.delete_many({})

menu.insert_many([
    # Kopi
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 60},
    {"nama_menu": "Kopi Tubruk", "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 80},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 70, "rekomendasi_barista": True},
    {"nama_menu": "Latte", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 65},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 90, "rekomendasi_barista": True},
    {"nama_menu": "V60 Single Origin", "harga": 32000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 30},
    # Makanan
    {"nama_menu": "Nasi Goreng Kampung", "harga": 28000, "kategori": "Makanan", "pedas_level": 2, "stok": 40},
    {"nama_menu": "Mie Goreng Pedas", "harga": 24000, "kategori": "Makanan", "pedas_level": 3, "stok": 45},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 55},
    {"nama_menu": "Kentang Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 1, "stok": 60},
    {"nama_menu": "Pisang Goreng", "harga": 14000, "kategori": "Makanan", "pedas_level": 0, "stok": 70},
    {"nama_menu": "Ayam Geprek", "harga": 27000, "kategori": "Makanan", "pedas_level": 3, "stok": 35},
])

print("Jumlah dokumen:", menu.count_documents({}))

Jumlah dokumen: 13


In [3]:
def tampil(judul, cursor):
    print(judul)
    for d in cursor:
        print(f"  - {d['nama_menu']} | {d['kategori']} | Rp{d['harga']:,}")
    print()

tampil("1. Menu Kopi dengan tingkat_kepahitan > 3:",
       menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}))

tampil("2. Menu dengan harga 15000 - 30000:",
       menu.find({"harga": {"$gte": 15000, "$lte": 30000}}))

tampil("3. Menu tanpa field pedas_level:",
       menu.find({"pedas_level": {"$exists": False}}))

1. Menu Kopi dengan tingkat_kepahitan > 3:
  - Espresso | Kopi | Rp18,000
  - Americano | Kopi | Rp20,000
  - Kopi Tubruk | Kopi | Rp12,000

2. Menu dengan harga 15000 - 30000:
  - Espresso | Kopi | Rp18,000
  - Americano | Kopi | Rp20,000
  - Cappuccino | Kopi | Rp25,000
  - Latte | Kopi | Rp26,000
  - Kopi Susu Gula Aren | Kopi | Rp22,000
  - Nasi Goreng Kampung | Makanan | Rp28,000
  - Mie Goreng Pedas | Makanan | Rp24,000
  - Roti Bakar Cokelat | Makanan | Rp15,000
  - Kentang Goreng | Makanan | Rp20,000
  - Ayam Geprek | Makanan | Rp27,000

3. Menu tanpa field pedas_level:
  - Espresso | Kopi | Rp18,000
  - Americano | Kopi | Rp20,000
  - Kopi Tubruk | Kopi | Rp12,000
  - Cappuccino | Kopi | Rp25,000
  - Latte | Kopi | Rp26,000
  - Kopi Susu Gula Aren | Kopi | Rp22,000
  - V60 Single Origin | Kopi | Rp32,000



In [4]:
menu.update_many({}, {"$set": {"status": "Tersedia"}})

menu.update_one({"nama_menu": "Kopi Tubruk"}, {"$set": {"status": "Habis"}})
menu.update_one({"nama_menu": "Nasi Goreng Kampung"}, {"$set": {"status": "Habis"}})

for d in menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {d['nama_menu']}: {d['status']}")

  - Espresso: Tersedia
  - Americano: Tersedia
  - Kopi Tubruk: Habis
  - Cappuccino: Tersedia
  - Latte: Tersedia
  - Kopi Susu Gula Aren: Tersedia
  - V60 Single Origin: Tersedia
  - Nasi Goreng Kampung: Habis
  - Mie Goreng Pedas: Tersedia
  - Roti Bakar Cokelat: Tersedia
  - Kentang Goreng: Tersedia
  - Pisang Goreng: Tersedia
  - Ayam Geprek: Tersedia


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType
from pyspark.sql.functions import avg

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

dokumen = list(menu.find({}))

skema = StructType([
    StructField("_id", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("harga", LongType(), True),
    StructField("stok", LongType(), True),
    StructField("tingkat_kepahitan", LongType(), True),
    StructField("pedas_level", LongType(), True),
    StructField("rekomendasi_barista", BooleanType(), True),
    StructField("status", StringType(), True),
])

baris = [(
    str(d["_id"]),
    d.get("nama_menu"),
    d.get("kategori"),
    d.get("harga"),
    d.get("stok"),
    d.get("tingkat_kepahitan"),
    d.get("pedas_level"),
    d.get("rekomendasi_barista"),
    d.get("status"),
) for d in dokumen]

df_menu = spark.createDataFrame(baris, schema=skema)

# 1. printSchema
df_menu.printSchema()
df_menu.show(truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 17:33:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)



+------------------------+-------------------+--------+-----+----+-----------------+-----------+-------------------+--------+
|_id                     |nama_menu          |kategori|harga|stok|tingkat_kepahitan|pedas_level|rekomendasi_barista|status  |
+------------------------+-------------------+--------+-----+----+-----------------+-----------+-------------------+--------+
|6ae31e1929b6879da15c2a55|Espresso           |Kopi    |18000|50  |5                |NULL       |NULL               |Tersedia|
|6ae31e1929b6879da15c2a56|Americano          |Kopi    |20000|60  |4                |NULL       |NULL               |Tersedia|
|6ae31e1929b6879da15c2a57|Kopi Tubruk        |Kopi    |12000|80  |4                |NULL       |NULL               |Habis   |
|6ae31e1929b6879da15c2a58|Cappuccino         |Kopi    |25000|70  |3                |NULL       |true               |Tersedia|
|6ae31e1929b6879da15c2a59|Latte              |Kopi    |26000|65  |2                |NULL       |NULL               |Te

In [6]:
# 2. Rata-rata harga per kategori
df_menu.groupBy("kategori").agg(avg("harga").alias("rata_rata_harga")).show()

spark.stop()

+--------+------------------+
|kategori|   rata_rata_harga|
+--------+------------------+
|    Kopi| 22142.85714285714|
| Makanan|21333.333333333332|
+--------+------------------+



Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame adalah ketidakseragaman field antar dokumen. Menu Kopi memiliki `tingkat_kepahitan`, menu Makanan memiliki `pedas_level`, dan hanya sebagian menu yang memiliki `rekomendasi_barista`, sedangkan Spark DataFrame membutuhkan skema tabel yang seragam dengan tipe data tetap pada setiap kolom. Selain itu, field `_id` bertipe `ObjectId` tidak dikenali Spark. Jika konversi dilakukan lewat pandas, nilai yang hilang menjadi `NaN` dan kolom bilangan bulat berubah menjadi desimal sehingga tipe datanya bergeser. Saya mengatasinya dengan mendefinisikan skema secara eksplisit menggunakan `StructType` yang memuat gabungan seluruh field, lalu mengisi field yang tidak ada dengan `None` memakai `d.get()` sehingga menjadi `null` di Spark. Field `_id` saya ubah menjadi string. Dengan cara ini tipe data setiap kolom terjamin dan hasil `printSchema()` serta `groupBy` berjalan benar.